In [1]:
# Google Colab Only
try:
    import google.colab  # noqa: F401

    # specify the version of DataEval (==X.XX.X) for versions other than the latest
    %pip install -q dataeval maite-datasets
except Exception:
    pass

In [2]:
from dataclasses import dataclass

import numpy as np
from IPython.display import display
from maite_datasets.image_classification import MNIST

from dataeval import Metadata
from dataeval.config import set_max_processes
from dataeval.data import Indices, View
from dataeval.protocols import DatasetMetadata, DatumMetadata
from dataeval.quality import Duplicates

set_max_processes(4)

In [3]:
# Load in the mnist dataset
testing_dataset = MNIST(root="./data/", image_set="test", download=True)

# Get the labels
labels = Metadata(testing_dataset).class_labels

In [4]:
# Creating some indices to duplicate
print("Exact duplicates")
duplicates = {}
for i in [1, 2, 5, 9]:
    matching_indices = np.where(labels == i)[0]
    print(f"\t{i} - ({matching_indices[23]}, {matching_indices[78]})")
    duplicates[int(matching_indices[78])] = int(matching_indices[23])

Exact duplicates
	1 - (180, 663)
	2 - (249, 728)
	5 - (219, 866)
	9 - (212, 773)


In [5]:
# Create a subset with the identified duplicate indices swapped
indices_with_duplicates = [duplicates.get(i, i) for i in range(len(testing_dataset))]
duplicates_ds = View(testing_dataset, Indices(indices_with_duplicates))

In [6]:
# Initialize the Duplicates class to begin to identify duplicate images.
identifyDuplicates = Duplicates()

# Evaluate the data
results = identifyDuplicates.evaluate(duplicates_ds)

/builds/jatic/aria/dataeval/src/dataeval/types/_output.py:270: FutureWarning: The default value of hash_radius will change from 0 in a future major release, so that Duplicates finds the perceptual near-duplicates it documents rather than only bit-identical ones (5 of 64 bits is the expected value). Pass hash_radius explicitly to pin the current behavior and silence this warning.
  result = fn(*args, **kwargs)


In [7]:
display(results)

shape: (133, 7)
┌──────────┬───────┬──────────┬──────────────┬────────────┬────────────┬───────────────────────┐
│ group_id ┆ level ┆ dup_type ┆ item_indices ┆ methods    ┆ differs_on ┆ annotation_divergence │
│ ---      ┆ ---   ┆ ---      ┆ ---          ┆ ---        ┆ ---        ┆ ---                   │
│ i64      ┆ str   ┆ str      ┆ list[i64]    ┆ list[str]  ┆ list[str]  ┆ f64                   │
╞══════════╪═══════╪══════════╪══════════════╪════════════╪════════════╪═══════════════════════╡
│ 0        ┆ item  ┆ exact    ┆ [180, 663]   ┆ ["xxhash"] ┆ null       ┆ null                  │
│ 1        ┆ item  ┆ exact    ┆ [212, 773]   ┆ ["xxhash"] ┆ null       ┆ null                  │
│ 2        ┆ item  ┆ exact    ┆ [219, 866]   ┆ ["xxhash"] ┆ null       ┆ null                  │
│ 3        ┆ item  ┆ exact    ┆ [249, 728]   ┆ ["xxhash"] ┆ null       ┆ null                  │
│ 4        ┆ item  ┆ near     ┆ [14, 8575]   ┆ ["dhash"]  ┆ null       ┆ null                  │
│ …        ┆ …

In [8]:
# TEST ASSERTION CELL ###
assert results.exact
assert len(results.exact) == len(duplicates)
for k, v in duplicates.items():
    assert [v, k] in results.exact

In [9]:
plan = results.deduplicate()
deduplicated_ds = View(duplicates_ds, Indices(plan, exclude=True))
print(plan)
print(f"{len(duplicates_ds)} items before, {len(deduplicated_ds)} after")

RemovalPlan(items=4)
10000 items before, 9996 after


In [10]:
# TEST ASSERTION CELL ###
assert len(plan) == len(duplicates)
assert len(deduplicated_ds) == len(duplicates_ds) - len(duplicates)

In [11]:
@dataclass
class BoxTarget:
    """A minimal object detection target: boxes, labels, and scores."""

    boxes: np.ndarray
    labels: np.ndarray
    scores: np.ndarray


class TinyObjectDetectionDataset:
    """Four items as two duplicate pairs: one relabeled, one augmented."""

    def __init__(self, images: list[np.ndarray], boxes: list[list[list[float]]], labels: list[list[int]]) -> None:
        self._images = images
        self._boxes = boxes
        self._labels = labels
        self.metadata: DatasetMetadata = DatasetMetadata(id="tiny-od", index2label={0: "object"})

    def __len__(self) -> int:
        return len(self._images)

    def __getitem__(self, index: int) -> tuple[np.ndarray, BoxTarget, DatumMetadata]:
        item_labels = np.asarray(self._labels[index], dtype=np.intp)
        target = BoxTarget(
            boxes=np.asarray(self._boxes[index], dtype=np.float32),
            labels=item_labels,
            scores=np.ones(len(item_labels), dtype=np.float32),
        )
        return self._images[index], target, DatumMetadata(id=index)


rng = np.random.default_rng(0)
same_image = rng.random((3, 16, 16))

# Items 0, 1: identical pixels, two different boxes drawn for the same collect - a divergent pair.
# Items 2, 3: identical annotation, two independently sampled images - an augmented pair.
annotation_images = [same_image, same_image.copy(), rng.random((3, 16, 16)), rng.random((3, 16, 16))]
annotation_boxes = [[[1.0, 1.0, 6.0, 6.0]], [[9.0, 9.0, 14.0, 14.0]], [[0.0, 0.0, 5.0, 5.0]], [[0.0, 0.0, 5.0, 5.0]]]
annotation_labels = [[0], [0], [0], [0]]

annotation_ds = TinyObjectDetectionDataset(annotation_images, annotation_boxes, annotation_labels)
annotation_results = Duplicates(hash_radius=0).evaluate(annotation_ds)

In [12]:
display(annotation_results.divergent())

shape: (1, 7)
┌──────────┬───────┬──────────┬──────────────┬────────────┬────────────────┬───────────────────────┐
│ group_id ┆ level ┆ dup_type ┆ item_indices ┆ methods    ┆ differs_on     ┆ annotation_divergence │
│ ---      ┆ ---   ┆ ---      ┆ ---          ┆ ---        ┆ ---            ┆ ---                   │
│ i64      ┆ str   ┆ str      ┆ list[i64]    ┆ list[str]  ┆ list[str]      ┆ f64                   │
╞══════════╪═══════╪══════════╪══════════════╪════════════╪════════════════╪═══════════════════════╡
│ 0        ┆ item  ┆ exact    ┆ [0, 1]       ┆ ["xxhash"] ┆ ["annotation"] ┆ 1.0                   │
└──────────┴───────┴──────────┴──────────────┴────────────┴────────────────┴───────────────────────┘

In [13]:
display(annotation_results.augmented())

shape: (1, 7)
┌──────────┬───────┬────────────┬──────────────┬────────────────┬─────────────┬────────────────────┐
│ group_id ┆ level ┆ dup_type   ┆ item_indices ┆ methods        ┆ differs_on  ┆ annotation_diverge │
│ ---      ┆ ---   ┆ ---        ┆ ---          ┆ ---            ┆ ---         ┆ nce                │
│ i64      ┆ str   ┆ str        ┆ list[i64]    ┆ list[str]      ┆ list[str]   ┆ ---                │
│          ┆       ┆            ┆              ┆                ┆             ┆ f64                │
╞══════════╪═══════╪════════════╪══════════════╪════════════════╪═════════════╪════════════════════╡
│ 1        ┆ item  ┆ annotation ┆ [2, 3]       ┆ ["annotation"] ┆ ["content"] ┆ null               │
└──────────┴───────┴────────────┴──────────────┴────────────────┴─────────────┴────────────────────┘

In [14]:
# TEST ASSERTION CELL ###
assert len(annotation_results.divergent()) == 1
assert annotation_results.divergent().data()["item_indices"].item().to_list() == [0, 1]
assert len(annotation_results.augmented()) == 1
assert annotation_results.augmented().data()["item_indices"].item().to_list() == [2, 3]